## B06阶段：修订处理并比较一个候选

追加到原项目，先运行B05基线。继承development、target、splits、evaluated、baseline_summary等，不重新划分、不替换主指标。若原项目接口不同，按原计划适配，不用新模板覆盖。

**事前比较问题、选择理由与保持条件：**（先填写）

使用给定的logistic候选，与原多数类基线比较；不要追加多个候选后择优。

In [ ]:
CANDIDATE = "logistic"  # 辅修固定候选，不改为运行多个模型后择优
assert CANDIDATE == "logistic"
required = [
    "development",
    "target",
    "splits",
    "evaluated",
    "baseline_summary",
    "PRIMARY_METRIC",
    "baseline_predictions",
    "SEED",
    "MODE",
    "baseline_protocol",
]
if not all(name in globals() for name in required):
    raise RuntimeError(
        "请先运行带baseline_protocol记录的B05基线；旧版片段需按原书面计划补记录"
    )
current_protocol = {
    "mode": MODE,
    "seed": int(SEED),
    "primary_metric": PRIMARY_METRIC,
    "split_indices": [(ti.tolist(), vi.tolist()) for ti, vi in splits],
}
if current_protocol != baseline_protocol:
    raise ValueError("B05后协议已改变；核对原计划并记录后重跑，不能只修改当前变量")
from copy import deepcopy

candidate_protocol = deepcopy(baseline_protocol)

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

project_features = ["planned_load", "temperature_setting", "material_score"]
project_X = development[project_features]
candidate = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1.0, max_iter=1000)),
    ]
)
candidate_predictions = np.full(len(target), -1, dtype=int)
candidate_coverage = np.zeros(len(target), dtype=int)
candidate_models = []
candidate_rows = []
for part, (ti, vi) in enumerate(splits, 1):
    fitted = clone(candidate).fit(project_X.iloc[ti], target[ti])
    candidate_models.append(fitted)
    np.testing.assert_allclose(
        fitted.named_steps["imputer"].statistics_,
        project_X.iloc[ti].median().to_numpy(),
    )
    pred = fitted.predict(project_X.iloc[vi])
    candidate_predictions[vi] = pred
    candidate_coverage[vi] += 1
    tn, fp, fn, tp = confusion_matrix(target[vi], pred, labels=[0, 1]).ravel()
    candidate_rows.append(
        {
            "part": part,
            "accuracy": (tp + tn) / len(vi),
            "recall": tp / (tp + fn) if tp + fn else np.nan,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }
    )
np.testing.assert_array_equal(candidate_coverage, evaluated.astype(int))
candidate_results = pd.DataFrame(candidate_rows)
display(candidate_results)
tn, fp, fn, tp = confusion_matrix(
    target[evaluated], candidate_predictions[evaluated], labels=[0, 1]
).ravel()
candidate_summary = {
    "accuracy": (tp + tn) / evaluated.sum(),
    "recall": tp / (tp + fn) if tp + fn else np.nan,
    "precision": tp / (tp + fp) if tp + fp else np.nan,
}
comparison = pd.DataFrame(
    [baseline_summary, candidate_summary], index=["baseline", CANDIDATE]
)
display(comparison)
print(
    "沿用主指标：",
    PRIMARY_METRIC,
    "；两方案主指标：",
    comparison[PRIMARY_METRIC].to_dict(),
)
print("评价行相同；仅开发内比较，未读取保留测试。NaN是未定义，不改挑其他主指标。")

**阶段解释：**

1. 引用主指标与TP/FP/FN/TN，说明候选相对基线的改变；没有改善也如实记录。
2. 写明填补/尺度fit来源、原划分与种子；为何没有先对全开发集fit？
3. 保留一个有依据的候选和局限，不追加搜索。上述各折模型供诊断，并非已经冻结的最终模型。

保存原项目Notebook，不另交报告；T15冻结，B08最终评价。